In [1]:
# imports!
from datasets import load_dataset
import pandas as pd
import os
import ast

In [28]:
# choose number of sentences
num_sen = 1000

# helper function
def make_prompt(row, dataset, include_answer=False, CoT=False):
    labels = ["A", "B", "C", "D"]
    if dataset == "hellaswag":
        question = row["ctx"].strip()
        choices = row["endings"]
        answer = ["A", "B", "C", "D"][int(row["label"])]

    elif dataset == "mmlu":
        question = row["question"].strip()
        choices = row["choices"]
        answer = ["A", "B", "C", "D"][int(row["answer"])]

    elif dataset == "mcqa":
        question = row["question"].strip()
        choices = [row[label] for label in labels]
        answer = row["alternative"]

    if isinstance(choices, str):
        choices = ast.literal_eval(choices)
    choices_text = "\n".join(f"{label}. {choice}" for label, choice in zip(labels, choices))

    prompt = f"{question}\n{choices_text}\nAnswer:"

    if CoT:
        prompt += " Let's think step by step."
    
    if include_answer:
        # if answer is stored as an integer: 0 -> A, 1 -> B, etc.
        if isinstance(answer, int):
            answer = labels[answer]

        prompt += f" {answer}"

    return prompt


# add prompt used for testing
def no_shot_prompt(df, dataset):
    df["prompt"] = df.apply(
        lambda row: make_prompt(row, dataset),
        axis=1,
    )
    return df

# few shot prompting
def make_few_shot_prompt(df, row, dataset, n_shot=3, seed=8):
    # row
    idx = row.name
    
    # candidate demonstrations exclude target
    candidates = df.drop(index=idx)

    # reproducible random selection
    demo_rows = candidates.sample(n=min(n_shot, len(candidates)), random_state=seed)

    demonstrations = [make_prompt(row, dataset, include_answer=True) for _, row in demo_rows.iterrows()]

    target = make_prompt(row, dataset, include_answer=False)
    fs_prompt = "\n\n".join(demonstrations + [target])

    return fs_prompt

# add few shot prompt used for testing
def few_shot_prompt(df, dataset, shots=3):
    df["prompt_fs"] = df.apply(
        lambda row: make_few_shot_prompt(df, row, dataset, n_shot=shots),
        axis=1,
    )
    return df

def cot_prompt(df, dataset):
    df["prompt_cot"] = df.apply(
        lambda row: make_prompt(row, dataset, CoT=True),
        axis=1,
    )
    return df

In [29]:
## == HELLASWAG == ##

# # import hellaswag
# hellaswag_dataset = load_dataset("Rowan/hellaswag")
# hellaswag_df = pd.DataFrame(hellaswag_dataset['validation']).sample(n=num_sen, random_state=8)

# # store
# hellaswag_df = no_shot_prompt(hellaswag_df, "hellaswag")
# hellaswag_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/hellaswag.csv", index=False)

hellaswag_df = pd.read_csv("~/TDA_RI/TDA_reason-interpret/datasets/hellaswag.csv")
hellaswag_df = few_shot_prompt(hellaswag_df, "hellaswag")
hellaswag_df = cot_prompt(hellaswag_df, "hellaswag")
hellaswag_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/hellaswag.csv", index=False)

In [30]:
hellaswag_size = len(hellaswag_df)
print("Size is:", hellaswag_size)
hellaswag_df.head()

print(hellaswag_df["prompt_fs"][1])
print(hellaswag_df["prompt_cot"][1])

Size is: 1000
This elderly man is standing up cutting the red tomato into slices. He is in the kitchen cutting the tomato using a cheap knife. it
A. takes awhile to get an image of the tomato he cut and the dried tomato they cut.
B. looked good on the bread.
C. takes 2 hours doing this before the tomato is fully cooked.
D. looks hard for him trying to use a knife that barely does anything to try and cut the tomato, but the difference shows in good products.
Answer: D

Children are picking up crabs in mud. the crabs
A. are being eaten by a peeler.
B. are cleaned off of the floor.
C. get stuck in the mud and the man jumps out pushing them away.
D. are put in a bucket.
Answer: D

A bottle of moisturizer is shown. A woman on a beach applies it to her skin. She rubs it into her skin. text
A. is shown shortly after.
B. pops up on the screen.
C. appears on screen describing the product.
D. is shown up against the screen.
Answer: C

A crowd watches as several runners in a race jog past. they a

In [31]:
## == MMLU DATASETS == ##

# # import mmlu_professional_law
# mmlu_law_dataset = load_dataset("cais/mmlu", "professional_law")
# mmlu_law_df = pd.DataFrame(mmlu_law_dataset['test']).sample(n=num_sen, random_state=8)
# mmlu_law_df = pd.DataFrame(mmlu_law_dataset["test"]).reset_index(names="ind").sample(n=num_sen, random_state=8)

# # store
# mmlu_law_df = no_shot_prompt(mmlu_law_df, "mmlu")
# mmlu_law_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_law.csv", index=False)

mmlu_law_df = pd.read_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_law.csv")
mmlu_law_df = few_shot_prompt(mmlu_law_df, "mmlu")
mmlu_law_df = cot_prompt(mmlu_law_df, "mmlu")
mmlu_law_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_law.csv", index=False)


# # import mmlu_moral_scenarios
# mmlu_moral_dataset = load_dataset("cais/mmlu", "moral_scenarios")
# mmlu_moral_df = pd.DataFrame(mmlu_moral_dataset['test']).reset_index(names="ind")

# # store
# mmlu_moral_df = no_shot_prompt(mmlu_moral_df, "mmlu")
# mmlu_moral_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_moral.csv", index=False)

mmlu_moral_df = pd.read_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_moral.csv")
mmlu_moral_df = few_shot_prompt(mmlu_moral_df, "mmlu")
mmlu_moral_df = cot_prompt(mmlu_moral_df, "mmlu")
mmlu_moral_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_moral.csv", index=False)


# # import mmlu_professional_psychology
# mmlu_psych_dataset = load_dataset("cais/mmlu", "professional_psychology")
# mmlu_psych_df = pd.DataFrame(mmlu_psych_dataset['test']).reset_index(names="ind")

# # store
# mmlu_psych_df = no_shot_prompt(mmlu_psych_df, "mmlu")
# mmlu_psych_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_psych.csv", index=False)

mmlu_psych_df = pd.read_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_psych.csv")
mmlu_psych_df = few_shot_prompt(mmlu_psych_df, "mmlu")
mmlu_psych_df = cot_prompt(mmlu_psych_df, "mmlu")
mmlu_psych_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_psych.csv", index=False)


# # import mmlu_elementary_mathematics
# mmlu_math_dataset = load_dataset("cais/mmlu", "elementary_mathematics")
# mmlu_math_df = pd.DataFrame(mmlu_math_dataset['test']).reset_index(names="ind")

# # store
# mmlu_math_df = no_shot_prompt(mmlu_math_df, "mmlu")
# mmlu_math_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_math.csv", index=False)

mmlu_math_df = pd.read_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_math.csv")
mmlu_math_df = few_shot_prompt(mmlu_math_df, "mmlu")
mmlu_math_df = cot_prompt(mmlu_math_df, "mmlu")
mmlu_math_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_math.csv", index=False)


# # import mmlu_miscellaneous
# mmlu_misc_dataset = load_dataset("cais/mmlu", "miscellaneous")
# mmlu_misc_df = pd.DataFrame(mmlu_misc_dataset['test']).reset_index(names="ind")

# # store
# mmlu_misc_df = no_shot_prompt(mmlu_misc_df, "mmlu")
# mmlu_misc_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_misc.csv", index=False)

mmlu_misc_df = pd.read_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_misc.csv")
mmlu_misc_df = few_shot_prompt(mmlu_misc_df, "mmlu")
mmlu_misc_df = cot_prompt(mmlu_misc_df, "mmlu")
mmlu_misc_df.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mmlu_misc.csv", index=False)

In [38]:
mmlu_law_size = len(mmlu_law_df)
print("law size is:", mmlu_law_size)
mmlu_law_df.head()

print(mmlu_law_df["prompt_fs"][1])
print(mmlu_law_df["prompt_cot"][1])

law size is: 1000
A driver of a car negligently hit a pedestrian. The pedestrian had an open wound and was rushed to the hospital for a blood transfusion. At the hospital she refused the transfusion because of religious beliefs. She died shortly thereafter. The physician verified that the patient would have fully recovered if she had agreed to treatment. What damages can the pedestrian's estate collect from the driver of the car?
A. All damages for wrongful death because it is foreseeable that this kind of negligence could injure someone who will not accept treatment for religious reasons.
B. No damages, because the pedestrian's wrongdoing in not accepting treatment was the sole cause for her death.
C. Damages for the wound and the pain and suffering up to the point that she refused the transfusion.
D. No damages, because the pedestrian's refusal of treatment constitutes contributory negligence.
Answer: C

A plaintiff sued an insurance company to recover life insurance proceeds. At tri

In [33]:
mmlu_moral_size = len(mmlu_moral_df)
print("moral size is:", mmlu_moral_size)
mmlu_moral_df.head()

moral size is: 895


,ind,question,subject,choices,answer,prompt,prompt_fs,prompt_cot
0,0,For which of these two scenarios does the main...,moral_scenarios,"['Wrong, Wrong', 'Wrong, Not wrong', 'Not wron...",3,For which of these two scenarios does the main...,For which of these two scenarios does the main...,For which of these two scenarios does the main...
1,1,For which of these two scenarios does the main...,moral_scenarios,"['Wrong, Wrong', 'Wrong, Not wrong', 'Not wron...",0,For which of these two scenarios does the main...,For which of these two scenarios does the main...,For which of these two scenarios does the main...
2,2,For which of these two scenarios does the main...,moral_scenarios,"['Wrong, Wrong', 'Wrong, Not wrong', 'Not wron...",3,For which of these two scenarios does the main...,For which of these two scenarios does the main...,For which of these two scenarios does the main...
3,3,For which of these two scenarios does the main...,moral_scenarios,"['Wrong, Wrong', 'Wrong, Not wrong', 'Not wron...",1,For which of these two scenarios does the main...,For which of these two scenarios does the main...,For which of these two scenarios does the main...
4,4,For which of these two scenarios does the main...,moral_scenarios,"['Wrong, Wrong', 'Wrong, Not wrong', 'Not wron...",3,For which of these two scenarios does the main...,For which of these two scenarios does the main...,For which of these two scenarios does the main...


In [8]:
mmlu_psych_size = len(mmlu_psych_df)
print("psych size is:", mmlu_psych_size)
mmlu_psych_df.head()

psych size is: 612


,ind,question,subject,choices,answer,prompt
0,0,Which of the following statements expresses a ...,professional_psychology,[Aging is related to an increase in vaginal lu...,3,Which of the following statements expresses a ...
1,1,"According to Piaget, children are ___________.",professional_psychology,"[“Blank slates”, Less intelligent than adults,...",2,"According to Piaget, children are ___________...."
2,2,If a test has a standard error of measurement ...,professional_psychology,[about 68% of the observed scores for the test...,1,If a test has a standard error of measurement ...
3,3,Any substance that can have a negative impact ...,professional_psychology,"[An Apgar, A teratogen, Only a problem in the ...",1,Any substance that can have a negative impact ...
4,4,If you were hired by a large company to develo...,professional_psychology,"[needs analysis., job evaluation., summative e...",0,If you were hired by a large company to develo...


In [34]:
mmlu_math_size = len(mmlu_math_df)
print("math size is:", mmlu_math_size)
mmlu_math_df.head()

math size is: 378


,ind,question,subject,choices,answer,prompt,prompt_fs,prompt_cot
0,0,What is the value of p in 24 = 2p?,elementary_mathematics,"['p = 4', 'p = 8', 'p = 12', 'p = 24']",2,What is the value of p in 24 = 2p?\nA. p = 4\n...,Ms. Perez drove a total of 40 miles in 5 days....,What is the value of p in 24 = 2p?\nA. p = 4\n...
1,1,Ms. Perez drove a total of 40 miles in 5 days....,elementary_mathematics,"['5', '7', '8', '9']",2,Ms. Perez drove a total of 40 miles in 5 days....,What is the value of p in 24 = 2p?\nA. p = 4\n...,Ms. Perez drove a total of 40 miles in 5 days....
2,2,Find the quotient of −40 ÷ (−8).,elementary_mathematics,"['1 over 5', '−5', '−1 over 5', '5']",3,Find the quotient of −40 ÷ (−8).\nA. 1 over 5\...,What is the value of p in 24 = 2p?\nA. p = 4\n...,Find the quotient of −40 ÷ (−8).\nA. 1 over 5\...
3,3,A soccer team has $90.00 to buy soccer balls. ...,elementary_mathematics,"['4', '5', '6', '7']",1,A soccer team has $90.00 to buy soccer balls. ...,What is the value of p in 24 = 2p?\nA. p = 4\n...,A soccer team has $90.00 to buy soccer balls. ...
4,4,You and three friends go to a concert. The tot...,elementary_mathematics,"['4t = 112; $448', '4t = 112; $28', 't over 4 ...",1,You and three friends go to a concert. The tot...,What is the value of p in 24 = 2p?\nA. p = 4\n...,You and three friends go to a concert. The tot...


In [10]:
mmlu_misc_size = len(mmlu_misc_df)
print("misc size is:", mmlu_misc_size)
mmlu_misc_df.head()

misc size is: 783


,ind,question,subject,choices,answer,prompt
0,0,An important source of information on the cred...,miscellaneous,"[the Retail Merchants Association, the local c...",2,An important source of information on the cred...
1,1,In preparation for a writing unit on short sto...,miscellaneous,"[Conferencing, Discipline-based inquiry, Self-...",1,In preparation for a writing unit on short sto...
2,2,Paper will burn at approximately what temperat...,miscellaneous,"[986 degrees, 2125 degrees, 3985 degrees, 451 ...",3,Paper will burn at approximately what temperat...
3,3,The Apple iMac computer is available in all of...,miscellaneous,"[Tangerine, Strawberry, Kiwi, Grape]",2,The Apple iMac computer is available in all of...
4,4,What were the first names of the early America...,miscellaneous,"[Morgan and Mason, Meriwether and William, Cab...",1,What were the first names of the early America...


In [36]:
## == MCQA == ##

# # import MCQA
# raw_MCQA = "https://raw.githubusercontent.com/C4AI/Pira/main/MCQA/MCQA-test.csv"
# df_MCQA = pd.read_csv(raw_MCQA)

# # store
# df_MCQA = no_shot_prompt(df_MCQA, "mcqa")
# df_MCQA.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mcqa.csv", index=False)

df_MCQA = pd.read_csv("~/TDA_RI/TDA_reason-interpret/datasets/mcqa.csv")
df_MCQA = few_shot_prompt(df_MCQA, "mcqa")
df_MCQA = cot_prompt(df_MCQA, "mcqa")
df_MCQA.to_csv("~/TDA_RI/TDA_reason-interpret/datasets/mcqa.csv", index=False)

In [39]:
MCQA_size = len(df_MCQA)
print("Size is:", MCQA_size)
df_MCQA.head()

print(df_MCQA["prompt_fs"][1])
print(df_MCQA["prompt_cot"][1])

Size is: 227
The article alludes to the fact that Brazil is still a hotspot for oil exploration, which factor led to this conclusion?
A. Campos 
B. The bidder was Shell and the block was Campos 
C. The fact that big firms of E&P participated on the auction for the blocks 
D. In BC-20 Block in the Campos Basin - Brazil, 110 kilometers away from Rio de Janeiro state coast. 
Answer: C

One particular block called more attention than the others even though there was only one bidder. Who was this bidder and what was the block called?
A. Campos 
B. The bidder was Shell and the block was Campos 
C. In Campos Basin. 
D. In BC-20 Block in the Campos Basin - Brazil, 110 kilometers away from Rio de Janeiro state coast. 
Answer: B

Why should the overburden be thin enough during dip-parallel steps?
A. Overcoming the challenges that have arisen with the discovery of Pre Salt offshore Brazil. 
B. A group related to recharge zones; a group formed by groundwater with high Cl/Br ratio; a group formed l